In [1]:
import napari
import os
from pathlib import Path
import numpy as np
import glob
from dask_image import imread
import yaml
import json
#from sahi import AutoDetectionModel, predict
import tqdm
import ngff_zarr as nz
import zarr
from napari.utils.transforms import Affine
import shutil
import transforms3d

from multiview_stitcher import msi_utils
from multiview_stitcher import spatial_image_utils as si_utils
from multiview_stitcher import fusion

import neuroglancer

In [ ]:
viewer = napari.Viewer()


## Importing data

In [6]:
root = r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721\20250721_selectedPos_1_Focus_Plane_order2_1'

viewer.layers.clear()
# list_of_files = [os.path.join(root, file) for file in os.listdir(root) if not 'converted' in file]
# _ = viewer.open(list_of_files, plugin='napari-autoscape')

In [7]:
folders = [os.path.join(root, folder) for folder in os.listdir(root) if os.path.isdir(os.path.join(root, folder))]
tif_files = glob.glob(os.path.join(root, '**', '*.tif'))
metadata = yaml.safe_load(open(os.path.join(root, folders[0], 'metadata.txt'), 'r'))['Summary']

In [8]:
positionlist = metadata['InitialPositionList']
positionlist = {position['Label']: position for position in positionlist}

## Stitching

In [10]:
tile_translations = []
tile_arrays = []
for label, position in tqdm.tqdm(positionlist.items()):
    z_scale = metadata['z-step_um'] if metadata['z-step_um'] != 0 else 1.0
    y_scale = metadata['PixelSize_um']
    x_scale = metadata['PixelSize_um']

    array = imread.imread(
        glob.glob(os.path.join(root, label, '*.tif'))[0]
    )

    #size_z = np.round(array.shape[0] * z_scale)
    size_y = np.round(array.shape[0] * y_scale)
    size_x = np.round(array.shape[1] * x_scale)

    tile_translations.append(
        {
            #"z": float(position["DeviceCoordinatesUm"]["ZStage:Z:32"][0]) - size_z // 2,
            "y": -float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][1]) - size_y // 2,
            "x": float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][0]) - size_x // 2
            }
    )
    tile_arrays.append(array)

100%|██████████| 323/323 [00:05<00:00, 57.60it/s]


In [11]:
y_min = min([translation['y'] for translation in tile_translations])
x_min = min([translation['x'] for translation in tile_translations])
y_max = max([translation['y'] + size_y for translation in tile_translations])
x_max = max([translation['x'] + size_x for translation in tile_translations])
print(f"y_min: {y_min}, y_max: {y_max}, x_min: {x_min}, x_max: {x_max}")

y_min: 3198.7138671875, y_max: 10344.7138671875, x_min: 41801.08203125, x_max: 48699.08203125


In [12]:
msims = []
for tile_array, tile_translation in zip(tile_arrays, tile_translations):
    sim = si_utils.get_sim_from_array(
        tile_array,
        dims=["c", "y", "x"],
        scale={"y": y_scale, "x": x_scale},
        translation=tile_translation,
        transform_key="stage_metadata",
    )
    msims.append(msi_utils.get_msim_from_sim(sim, scale_factors=[]))

fused_sim = fusion.fuse(
    [msi_utils.get_sim_from_msim(msim) for msim in msims],
    transform_key="stage_metadata",
    output_chunksize={'c': 1, 'y': 1528, 'x': 1528}
).squeeze().pad(x=1, y=1, constant_values=0)

In [13]:
ngff_image = nz.to_ngff_image(
    fused_sim.data.compute(),
    dims=["y", "x"],
    translation={
        "y": float(y_min),
        "x": float(x_min),
    },
    scale={"y": float(y_scale), "x": float(x_scale)},
    axes_units={"y": "micrometer", "x": "micrometer"},
    )

In [14]:
ngff_multiscales = nz.to_multiscales(
    ngff_image,
    scale_factors=[2, 4, 8, 16])

In [15]:
target = root + 'ome.zarr'
#nz.to_ngff_zarr(target, ngff_multiscales1, version='0.6.dev1')
# nz.to_ngff_zarr(target, multiscales=ngff_multiscales, version="0.5")

## SCAPE visualization

In [16]:
import os
import json
from skimage import io
import numpy as np
import napari

root = r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721\SCAPE\20250721_Pos1-2'

metadata_path = os.path.join(root, '20250721_Pos1-2_MMStack_Pos0_metadata.txt')
with open(metadata_path, 'r', encoding='latin-1') as f:
    json_data = json.load(f)

x = float(json_data['Summary']['Position_X'].replace(',', '').replace(' µm', '').replace('µm', ''))
y = -float(json_data['Summary']['Position_Y'].replace(',', '').replace(' µm', '').replace('µm', ''))
angle = float(json_data['Summary']['StageScanAnglePathA'])
pixelsize_xy = float(json_data['Summary']['PixelSize_um'])
z_step = float(json_data['Summary']['z-step_um'])

slices_per_volume = int(json_data['Summary']['Slices'])

image_name = metadata_path.replace('_metadata.txt', '.ome.tif')
SCAPE_stack = io.imread(image_name)[0]

In [17]:
ngff_image = nz.to_ngff_image(
    SCAPE_stack,
    dims=["z", "y", "x"],
    translation={
        "z": 0.0,
        "y": y,
        "x": x,
    },
    scale={"z": float(z_step), "y": float(pixelsize_xy), "x": float(pixelsize_xy)},
    axes_units={"z": "micrometer", "y": "micrometer", "x": "micrometer"},
    )

ngff_multiscales = nz.to_multiscales(
    ngff_image,
    scale_factors=[2, 4, 8])

target = metadata_path.replace('_metadata.txt', '.ome.zarr')
nz.to_ngff_zarr(target, multiscales=ngff_multiscales, version="0.5")

In [25]:
neuroglancer.__version__

AttributeError: module 'neuroglancer' has no attribute '__version__'

### Calculate affine matrix

In [18]:
image_name = metadata_path.replace('_metadata.txt', '.ome.tif')
SCAPE_stack = io.imread(image_name)[0]

size_y = SCAPE_stack.shape[1] * pixelsize_xy
size_x = SCAPE_stack.shape[2] * pixelsize_xy

shear_factor = np.cos(angle * np.pi / 180.0) * (z_step / pixelsize_xy)
new_dz = np.sin(angle * np.pi / 180.0) * z_step

shear_matrix = np.eye(3)
shear_matrix[1, 0] = shear_factor

affine_matrix = Affine(
    shear=shear_matrix,
)
scale = np.array([z_step, pixelsize_xy, pixelsize_xy])
scale_correction = np.array([np.sin(angle * np.pi / 180.0), 1, 1])
translate = [0, y + size_y / 2, x - size_x / 2]

# rotate to correct viewing angle
rotation_matrix = transforms3d.euler.euler2mat(angle, 0, 0)

## Display in neuroglancer

In [50]:
# full affine matrix
affine_matrix_ng = Affine(
    shear=shear_matrix,
    scale = scale_correction * scale,
    rotate = rotation_matrix,
    translate=translate
)

array([0.37735479, 0.391     , 0.391     ])

In [65]:
viewer_n = neuroglancer.Viewer()
viewer_n

http://127.0.0.1:49871/v/4acb1adfc10384b2fd6efe9340b28875814aadc0/

In [82]:
np.asarray(affine_matrix_ng)

array([[ 3.77354790e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 9.51621476e-02,  1.17531685e-01,  3.72917287e-01,
         5.99950000e+03],
       [-3.01940792e-01, -3.72917287e-01,  1.17531685e-01,
         4.41321000e+04],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [85]:
transform = neuroglancer.CoordinateSpaceTransform(
    output_dimensions=neuroglancer.CoordinateSpace(
        names=["z", "y", "x"],
        units="µm",
        scales=scale_correction * scale
    ),
    matrix=np.asarray(affine_matrix_ng)[:3]
)

source = neuroglancer.LayerDataSource(
    "zarr://http://192.168.178.64:3000/SCAPE/20250721_Pos1-2/20250721_Pos1-2_MMStack_Pos0.ome.zarr/",
    transform=transform
)

with viewer_n.txn() as s:
    s.layers['scape'] = neuroglancer.ImageLayer(
        source=source,
    )


In [43]:
with viewer_n.txn() as s:
    print(dir(s.layers['overview'].layer))

['VolumeRendering', '__annotations__', '__class__', '__deepcopy__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '_cached_wrappers', '_get_wrapped', '_json_data', '_lock', '_readonly', '_set_wrapped', 'annotationColor', 'annotation_color', 'blend', 'crossSectionRenderScale', 'cross_section_render_scale', 'interpolate', 'localDimensions', 'localPosition', 'localVelocity', 'local_dimensions', 'local_position', 'local_velocity', 'opacity', 'panels', 'pick', 'shader', 'shaderControls', 'shader_controls', 'source', 'supports_readonly', 'supports_validation', 'tab', 'to_json', 'tool', 'tool_bindings', 'type', 'volumeRenderingDepthSamples', 'volumeRenderingGain', 'volumeRenderingMode', 'volume_rendering', 'vol

### Store with ome-zarr-py

In [ ]:
from ome_zarr.io import parse_url
from ome_zarr.format import FormatV06
from ome_zarr.writer import write_image, add_metadata

root = r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721'
path = os.path.join(root, 'scape_stack.ome.zarr')

store = parse_url(path, mode="w").store
root = zarr.group(store=store)

transformations = [
    {
        "type": "sequence",
        "input": "physical",
        "output": "world",
        "transformations": [
            {
                "type": "rotation",
                "rotation": np.asarray(rotation_matrix).tolist()
            },
            {
                "type": "affine",
                "affine": np.asarray(affine_matrix).tolist()
            },
            {
                "type": "scale",
                "scale": scale_correction.tolist()
            },
            {
                "type": "translate",
                "translation": translate
            }
        ]
    }
]

shutil.rmtree(path, ignore_errors=True)
write_image(
    SCAPE_stack,
    group=root, axes=['z', 'y', 'x'],
    )

### Display in napari

In [ ]:
viewer.open(
    r'C:\Users\johan\Desktop\local_data\BiaPoL\autoSCAPE\20250721\20250721_selectedPos_1_Focus_Plane_order2_1ome.zarr',
    plugin='napari-ome-zarr')

layer = viewer.add_image(
    SCAPE_stack,
    name='SCAPE Stack',
    contrast_limits=[0, 3000],
    colormap='bop blue',
    blending='additive',
    axis_labels=['z', 'y', 'x'],
    affine=affine_matrix,
)

layer.bounding_box.visible = True
viewer.camera.angles = (0, 0, 90)
viewer.camera.center = (0, y + size_y / 2, x - size_x / 2)

Root group {'ome': {'version': '0.5', 'multiscales': [{'axes': [{'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}], 'datasets': [{'path': 'scale0/image', 'coordinateTransformations': [{'scale': [0.3245, 0.3245], 'type': 'scale'}, {'translation': [3198.7138671875, 41801.08203125], 'type': 'translation'}]}, {'path': 'scale1/image', 'coordinateTransformations': [{'scale': [0.649, 0.649], 'type': 'scale'}, {'translation': [3198.8761171875, 41801.24428125], 'type': 'translation'}]}, {'path': 'scale2/image', 'coordinateTransformations': [{'scale': [1.298, 1.298], 'type': 'scale'}, {'translation': [3199.2006171875, 41801.568781250004], 'type': 'translation'}]}, {'path': 'scale3/image', 'coordinateTransformations': [{'scale': [2.596, 2.596], 'type': 'scale'}, {'translation': [3199.8496171875, 41802.21778125], 'type': 'translation'}]}, {'path': 'scale4/image', 'coordinateTransformations': [{'scale': [5.192, 5.192], 'type': 'scale'}, {'tra

## Object detection

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path='.runs/detect/train4/weights/yolo11n.pt',
    confidence_threshold=0.3,
    device="cuda:0",  # or 'cuda:0'
)

In [19]:
data = ngff_multiscales.images[0].data[1:-2, 1:-2]

# normalize to mean=126, std=64
data_normalized = (data - data.mean()) / data.std()  # standardize to mean=0, std=1
data = data_normalized * 64 + 126  # scale to mean=126, std=64
data = np.clip(data, 0, 255).astype(np.uint8)  # clip to valid range and convert to uint8
data = np.stack([data, data, data], axis=-1).compute().astype(np.uint8)

In [20]:
result = predict.get_prediction(data, detection_model=detection_model)

In [21]:
result.object_prediction_list

[]

In [ ]:
rectangles = []
features = pd.DataFrame()
for layer in viewer.layers:
    if not isinstance(layer, layers.Image):
        continue

    io.imsave('image.png', layer.data[0][0].compute())
    prediction = model('image.png')
    for obj_class, box, conf in zip(prediction[0].boxes.cls, prediction[0].boxes.xywhn, prediction[0].boxes.conf):
        cls = int(obj_class)
        box = box.detach().cpu().numpy()

        z0 = layer.translate[0]
        y0 = layer.translate[1]
        x0 = layer.translate[2]
        img_size = layer.data.shape * layer.scale
        y0_box = box[1] * img_size[1]
        x0_box = box[0] * img_size[2]
        w = box[2] * img_size[2]
        h = box[3] * img_size[1]

        rectangle_corners = np.array([
            [z0, y0 + y0_box - h/2, x0 + x0_box - w/2],
            [z0, y0 + y0_box - h/2, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box - w/2]
        ])
        rectangle_corners[:, 1:] += box[:2] * layer.scale[1:]

        _features = pd.DataFrame({
            'y0': [y0_box],
            'x0': [x0_box],
            'w': [w],
            'h': [h],
            'cls': [cls],
            'conf': [conf.detach().cpu().numpy()],
            'area': [w * h],
        })
        features = pd.concat([features, _features], ignore_index=True)

    rectangles.append(rectangle_corners)